# The lifecycle, and what it refuses

§5.2.6 draws a diagram, and `ddn.model.lifecycle` is a transcription of it —
written from the document rather than from what any other module happens to do
with `status`, so that it can disagree with the code if the code is wrong. A
transition the diagram does not draw does not exist: `IllegalTransition` is a
`ValueError`, not a warning.

**Unlike the other notebooks, nothing here is a placeholder.** No rate is
invented and no number is read off §10's worked example; every statement below
is about the document. Where the diagram is drawn rather than stated, the
module's docstring records the reading it took — three of them survive, and
three more were retired when v0.13 drew the edges they had argued for.

What this notebook shows is mostly the refusals, because those are what a state
machine is for. A machine that accepted everything would pass every test that
walks a legal path.

In [ ]:
from datetime import date
from itertools import pairwise
from pathlib import Path

from ddn.lastmile.plan import expired, retryable, sweep_expired, withdraw
from ddn.model import (
    TERMINAL,
    TRANSITIONS,
    IllegalTransition,
    Status,
    advance,
    after_attempt,
    settles,
)
from ddn.model.lifecycle import ATTEMPT_OUTCOMES, CANCELLED, after_cancellation

TODAY = date(2026, 9, 16)

edges = sum(len(reachable) for reachable in TRANSITIONS.values())
print(f"{len(TRANSITIONS)} statuses, {edges} edges")
print("terminal:", ", ".join(sorted(TERMINAL)))

# Quoted in the paragraph under the next cell. §5.2.6 is edited more often than
# any other section — it gained three edges at v0.13 alone — so the two counts
# are held to the table rather than typed into the sentence.
assert (len(TRANSITIONS), edges) == (18, 26)

Eighteen statuses and twenty-six edges, two of them terminal.

The graph is small enough to walk by hand and the walking is the point: every
stage of §5 hands an envelope on by *naming the next status*, and the naming
is validated in one place.

In [ ]:
happy = [Status.REQUESTED, Status.COLLECTED, Status.RECEIVED_AT_HUB,
         Status.RECONCILED, Status.SORTED, Status.READY, Status.LINE_HAUL,
         Status.AT_DEPOT, Status.DISPATCHED, Status.DELIVERED]

for source, target in pairwise(happy):
    print(f"  {source:<18} -> {advance(source, target)}")

## What it refuses

Each of these is a move some module could plausibly want to make, and §5.2.6
draws none of them. The message names what *is* drawn, because an error that
only says no makes the caller go and read the diagram.

- `Ready -> At depot` skips the line-haul that put it there. §5.2.6
  parenthesises `(Line-haul → At depot)` — the depot leg is optional, so
  §3.3's hub-direct envelopes go straight to Dispatched — but "optional" is
  not "silent".
- `Delivered -> Return run` is a cancellation that arrived after the stop was
  visited. §6 refuses it there too — "Refused once Delivered, which is
  terminal (§5.2.6)" — and `ddn/api/events.py` renders that refusal as a `409`
  carrying the state it refused.
- `In transfer -> Dispatched` is §7.1's "an envelope in transfer is not
  routable until it arrives at the destination depot", as a missing edge.
- `Sorted -> Dispatched` skips Ready, which §5.2.6 makes the only routable
  status.

In [ ]:
refused = ((Status.READY, Status.AT_DEPOT),
           (Status.DELIVERED, Status.RETURN_RUN),
           (Status.IN_TRANSFER, Status.DISPATCHED),
           (Status.SORTED, Status.DISPATCHED))

for source, target in refused:
    try:
        advance(source, target)
    except IllegalTransition as refusal:
        print(" ", refusal)
    else:
        raise AssertionError(f"§5.2.6 draws no {source} -> {target}")

# Every §-quote in the paragraph above, read out of the document rather than
# out of what this notebook remembers of it. A quotation mark is a claim.
# Anchored on the repository root, not on the working directory: the suite
# executes this from the root and `make notebooks` opens Jupyter rooted at
# `notebooks/`, so a relative path is only ever right for one of the two.
DOC = "docs/vrp-problem-definition.md"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / DOC).exists())
SPEC = (ROOT / DOC).read_text(encoding="utf-8")
QUOTED = ("(Line-haul → At depot)",
          "Refused once Delivered, which is terminal (§5.2.6)",
          ("an envelope in transfer is not routable until it arrives at "
           "the destination depot"))
for quoted in QUOTED:
    assert quoted in SPEC, f"§ has been reworded: {quoted!r}"


## §6's outcomes, and the one that is not an outcome

A delivery attempt produces one of four outcomes and `after_attempt` maps each
to the status §5.2.6 gives it. The map goes through `advance` rather than
returning `Status(outcome)`: the two spellings coincide today, and a §5.2.6
change that broke the coincidence should surface here rather than in a pool
three stages later.

`settles` asks whether §6 is finished. Only one of the four is.

`Cancelled` is §6's fifth row and it is **not** an attempt outcome — nothing
was attempted. Asking for it here raises, which is the same refusal as any
other undrawn edge.

In [ ]:
for outcome in sorted(ATTEMPT_OUTCOMES):
    status = after_attempt(outcome)
    print(f"  {outcome:<10} -> {status:<12}"
          f"{'§6 is done with it' if settles(outcome) else 'tomorrow carries it'}")

try:
    after_attempt(CANCELLED)
except IllegalTransition as refusal:
    print(f"\n  {CANCELLED}:", refusal)

assert len(ATTEMPT_OUTCOMES) == 4 and CANCELLED not in ATTEMPT_OUTCOMES
assert [o for o in sorted(ATTEMPT_OUTCOMES) if settles(o)] == ["Delivered"]

### A withdrawal reaches the return run from both sides of dispatch

§6 v0.15 lets an envelope be withdrawn before it is dispatched — it is still
Ready — or after, when "the stop is removed at the next plan refresh provided
it has not been visited". Two sources, one destination, and `after_cancellation`
validates from wherever the envelope actually is rather than from an assumed
source.

Which is why the third line below is a refusal and not a special case: it is
`advance` declining an edge, not a rule written a second time.

In [ ]:
landed = {}
for status in (Status.READY, Status.DISPATCHED, Status.DELIVERED):
    try:
        landed[status] = str(after_cancellation(status))
    except IllegalTransition:
        landed[status] = "refused -> the API answers 409"
    print(f"  withdrawn while {status:<12} -> {landed[status]}")

assert landed == {Status.READY: "Return run", Status.DISPATCHED: "Return run",
                  Status.DELIVERED: "refused -> the API answers 409"}

assert ("the stop is removed at the next plan refresh provided it has not "
        "been visited") in SPEC


## §6.1's clock is two questions, not one

`expired` and `retryable` look like the same test and are asked at different
moments about different things.

`expired` is asked at **dispatch**: is the SLA date already behind us? §6.1
gives the clock — "retried until its **SLA date**" — and §7.1 makes the far
side of it a hard constraint: "An envelope is not dispatched for delivery after
its SLA date." So this runs before §8's capacity decision; an expired envelope that
reached `select` would compete for a bike it is not allowed to board.

`retryable` is asked at the **end of the day**, about an envelope that was
postponed: is there a tomorrow for it? An envelope whose SLA date is today is
not expired and is not retryable, and that one row is the whole difference.

In [ ]:
pool = [{"package_id": "P1", "sla_date": "2026-09-15"},   # yesterday
        {"package_id": "P2", "sla_date": "2026-09-16"},   # today
        {"package_id": "P3", "sla_date": "2026-09-17"},   # tomorrow
        {"package_id": "P4"}]                             # §9.1 makes it optional

print(f"{'':<5}{'sla':<13}{'expired':<9}{'retryable':<10}")
answers = []
for envelope in pool:
    gone, again = expired(envelope, TODAY), retryable(envelope, TODAY)
    answers.append((gone, again))
    print(f"{envelope['package_id']:<5}{envelope.get('sla_date', '—'):<13}"
          f"{gone!s:<9}{again!s:<10}")

# P2 is the row the paragraph above is about, and the only one where the two
# answers differ from each other's negation.
assert answers == [(True, False), (False, False), (False, True), (False, True)]

live, going_back = sweep_expired(pool, TODAY)
print(f"\n§6.1 sweep: {[e['package_id'] for e in live]} may be dispatched, "
      f"{[e['package_id'] for e in going_back]} is tonight's §5.5 load")

# §6 v0.15's withdrawal is a pool operation, and it reports what it actually
# removed rather than what it was asked to: P9 is not this facility's, and an
# operator not told so believes an envelope is stopped when it is on a bike
# somewhere else.
kept, removed = withdraw(live, ["P2", "P9"])
print(f"withdrawing P2 and P9: kept {[e['package_id'] for e in kept]}, "
      f"removed {list(removed)}")
assert removed == ("P2",)

for quoted in ("retried until its **SLA date**",
               ("An envelope is not dispatched for delivery after "
                "its SLA date.")):
    assert quoted in SPEC, f"§ has been reworded: {quoted!r}"


## The shape of the whole thing

Two properties worth having, neither of them stated in §5.2.6 and both of them
things the diagram would be wrong without: every status is reachable from
`Requested`, and every status can reach a terminal one.

The second is the one that matters operationally. A status that could not reach
`Delivered` or `Returned to customer` would be a place envelopes accumulate,
and §5.6's next day would inherit a pool that never drains.

In [ ]:
seen, frontier = {Status.REQUESTED}, [Status.REQUESTED]
while frontier:
    for target in TRANSITIONS[frontier.pop()]:
        if target not in seen:
            seen.add(target)
            frontier.append(target)


def ends(status, walked=frozenset()):
    """Whether §5.2.6 lets an envelope here ever leave the system."""
    if status in TERMINAL:
        return True
    return status not in walked and any(
        ends(target, walked | {status}) for target in TRANSITIONS[status])


stuck = sorted(s for s in TRANSITIONS if not ends(s))
print(f"reachable from Requested: {len(seen)} of {len(TRANSITIONS)}")
print(f"statuses with no way out: {stuck or 'none'}")

assert seen == set(TRANSITIONS), "a status nothing reaches is a status nothing uses"
assert not stuck, "§5.6 would inherit a pool that never drains"